# Practice: one neuron, on real data
*AI from scratch, chapter 3* · lesson: https://ihor.sh/ai/03-a-neuron.html

A single neuron with a sigmoid is **logistic regression**, still one of the most used models at work. Train one
from scratch on 569 real tumor measurements, compare it with scikit-learn's, see it fail on XOR, then write it
in PyTorch. Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Compute elementwise cross-entropy: y * np.log(p) + (1 - y) * np.log(1 - p).', 'Return negative mean across all examples: -np.mean(...).'], 2: ['Compute predicted probabilities: p = sigmoid(X @ w + b).', 'Compute prediction error residuals: err = p - y.', 'Compute grad_w = X.T @ err / len(y) and grad_b = err.mean().'], 3: ['Run 500 gradient descent iterations: Compute gw, gb = gradients(w, b, Xtr, ytr). Update w = w - 0.5 * gw and b = b - 0.5 * gb. Append log_loss(sigmoid(Xtr @ w + b), ytr) to losses.', 'Evaluate accuracy on test set: test_acc = ((sigmoid(Xte @ w + b) > 0.5) == yte).mean().'], 4: ['Fit LogisticRegression(max_iter=1000).fit(Xtr, ytr).', 'Compute test accuracy score sk_acc = sk.score(Xte, yte).'], 5: ['Initialize random weights wx of shape (2,) and scalar bias bx = 0.0.', 'Train with 2000 steps of gradient descent with lr=0.5.', 'Compute final prediction accuracy: xor_acc = ((sigmoid(Xx @ wx + bx) > 0.5) == yx).mean().'], 6: ['Run 500 training steps: opt.zero_grad(), loss = loss_fn(model(Xt).squeeze(1), yt), loss.backward(), opt.step().', 'Evaluate accuracy on test set inside with torch.no_grad().']}
_ANSWERS = {1: 'cmV0dXJuIC1ucC5tZWFuKHkgKiBucC5sb2cocCkgKyAoMSAtIHkpICogbnAubG9nKDEgLSBwKSk=', 2: 'ZXJyID0gc2lnbW9pZChYIEAgdyArIGIpIC0geQpyZXR1cm4gWC5UIEAgZXJyIC8gbGVuKHkpLCBlcnIubWVhbigp', 3: 'Zm9yIHN0ZXAgaW4gcmFuZ2UoNTAwKToKICAgIGd3LCBnYiA9IGdyYWRpZW50cyh3LCBiLCBYdHIsIHl0cikKICAgIHcsIGIgPSB3IC0gMC41ICogZ3csIGIgLSAwLjUgKiBnYgogICAgbG9zc2VzLmFwcGVuZChsb2dfbG9zcyhzaWdtb2lkKFh0ciBAIHcgKyBiKSwgeXRyKSkKdGVzdF9hY2MgPSAoKHNpZ21vaWQoWHRlIEAgdyArIGIpID4gMC41KSA9PSB5dGUpLm1lYW4oKQ==', 4: 'c2sgPSBMb2dpc3RpY1JlZ3Jlc3Npb24obWF4X2l0ZXI9MTAwMCkuZml0KFh0ciwgeXRyKQpza19hY2MgPSBzay5zY29yZShYdGUsIHl0ZSk=', 5: 'd3gsIGJ4ID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKDApLm5vcm1hbCgwLCAwLjEsIDIpLCAwLjAKZm9yIF8gaW4gcmFuZ2UoMjAwMCk6CiAgICBndywgZ2IgPSBncmFkaWVudHMod3gsIGJ4LCBYeCwgeXgpCiAgICB3eCwgYnggPSB3eCAtIDAuNSAqIGd3LCBieCAtIDAuNSAqIGdiCiAgICB4b3JfYWNjID0gKChzaWdtb2lkKFh4IEAgd3ggKyBieCkgPiAwLjUpID09IHl4KS5tZWFuKCk=', 6: 'Zm9yIHN0ZXAgaW4gcmFuZ2UoNTAwKToKICAgIG9wdC56ZXJvX2dyYWQoKQogICAgbG9zcyA9IGxvc3NfZm4obW9kZWwoWHQpLnNxdWVlemUoMSksIHl0KQogICAgbG9zcy5iYWNrd2FyZCgpCiAgICBvcHQuc3RlcCgp'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## 1. The data
The Wisconsin breast-cancer dataset (bundled with scikit-learn): 30 measurements of cell nuclei per tumor, label
1 = benign, 0 = malignant. Standardize the features (z-scores, probability 2) and hold out 25% for testing.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
d = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(d.data, d.target, test_size=0.25, random_state=0)
mu, sd = Xtr.mean(0), Xtr.std(0)
Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd            # scale with the training set's numbers only
print(Xtr.shape, "benign share:", ytr.mean().round(3))

## 2. The loss and its gradient
Write `log_loss(p, y)`: the mean of $-[y \ln p + (1-y)\ln(1-p)]$, and `gradients(w, b, X, y)` returning
$\partial L/\partial w = \frac1n X^\top(p - y)$ and $\partial L/\partial b = \text{mean}(p - y)$ with $p = \sigma(Xw + b)$.

In [ ]:
def log_loss(p, y):
    """Compute binary cross-entropy (log loss).
    
    Formula: -mean(y * ln(p) + (1 - y) * ln(1 - p))
    
    Args:
        p: 1D NumPy array of predicted probabilities in (0, 1).
        y: 1D NumPy array of true binary labels (0 or 1).
    Returns:
        float: Average binary cross-entropy loss.
    """
    # YOUR CODE HERE (stuck? hint(1), then answer(1))
    ...

def gradients(w, b, X, y):
    """Compute gradients of binary cross-entropy loss with respect to weights w and bias b.
    
    Formula:
        p = sigmoid(X @ w + b)
        err = p - y
        grad_w = (1 / N) * X.T @ err
        grad_b = mean(err)
        
    Args:
        w: 1D NumPy array of weights (shape D).
        b: Float scalar bias.
        X: 2D NumPy array of input features (shape N, D).
        y: 1D NumPy array of target labels (shape N).
    Returns:
        tuple: (grad_w, grad_b) where grad_w has shape (D,) and grad_b is float scalar.
    """
    # YOUR CODE HERE (stuck? hint(2), then answer(2))
    ...

In [ ]:
check(2, lambda: np.isclose(log_loss(np.array([0.9, 0.2]), np.array([1, 0])), -(np.log(0.9) + np.log(0.8)) / 2), 
      "log_loss should evaluate average negative log probability assigned to ground truth labels.")
def _num(w, b, X, y, j, h=1e-6):
    e = np.zeros_like(w); e[j] = h
    return (log_loss(sigmoid(X @ (w + e) + b), y) - log_loss(sigmoid(X @ (w - e) + b), y)) / (2 * h)
w0 = np.random.default_rng(1).normal(0, 0.1, 30)
check(2, lambda: np.isclose(gradients(w0, 0.0, Xtr, ytr)[0][3], _num(w0, 0.0, Xtr, ytr, 3), rtol=1e-4), 
      "Analytical gradient ∂L/∂w[3] must match central numerical finite difference derivative.")

## 3. Train it
500 steps of gradient descent with learning rate 0.5 from all-zero weights. Record the loss each step. Then set
`test_acc`: the share of test tumors classified right (predict 1 when $p > 0.5$).

In [ ]:
w, b, losses = np.zeros(30), 0.0, []
test_acc = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
plt.plot(losses); plt.xlabel("step"); plt.ylabel("log loss"); plt.show()
print("test accuracy:", test_acc)

In [ ]:
check(3, lambda: test_acc > 0.95, 
      "Logistic regression test accuracy on breast cancer dataset should exceed 95%.")

## 4. Compare with scikit-learn
`LogisticRegression` is the same model, trained with a smarter optimizer and a little L2 regularization.
Fit it and set `sk_acc` (its test accuracy). Which features got the biggest weights (in absolute value)?

In [ ]:
from sklearn.linear_model import LogisticRegression
sk_acc = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("scikit-learn:", sk_acc)
print("biggest weights:", list(np.array(d.feature_names)[np.argsort(-abs(w))[:5]]))

In [ ]:
check(4, lambda: abs(sk_acc - test_acc) < 0.03, 
      "Scikit-learn LogisticRegression accuracy should match your custom neuron within 3 percentage points.")

## 5. Where one neuron fails: XOR
Train the same neuron on the four XOR points for 2000 steps. Set `xor_acc`. One straight cut can't put opposite
corners together (the lesson proves it), so it stays stuck around half right.

In [ ]:
Xx = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float); yx = np.array([0, 1, 1, 0])
xor_acc = None
# YOUR CODE HERE (stuck? hint(5), then answer(5))
...
print("XOR accuracy:", xor_acc)

In [ ]:
check(5, lambda: xor_acc is not None and xor_acc <= 0.75, 
      "A single linear neuron cannot solve XOR; accuracy cannot exceed 75% (at most 3 of 4 points).")

## 6. The neuron in PyTorch
`nn.Linear(30, 1)` holds w and b; `BCEWithLogitsLoss` is the log loss with the sigmoid built in (more stable).
Train it with `torch.optim.SGD(model.parameters(), lr=0.5)` for 500 steps: zero the gradients, compute the loss,
`.backward()`, `.step()`. Set `torch_acc`.

In [ ]:
import torch
from torch import nn
torch.manual_seed(0)
Xt, yt = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.float32)
model = nn.Linear(30, 1)
opt = torch.optim.SGD(model.parameters(), lr=0.5)
loss_fn = nn.BCEWithLogitsLoss()
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...
with torch.no_grad():
    torch_acc = ((model(torch.tensor(Xte, dtype=torch.float32)).squeeze(1) > 0).numpy() == yte).mean()
print("PyTorch:", torch_acc)

In [ ]:
check(6, lambda: torch_acc > 0.95, 
      "PyTorch single-neuron model should achieve > 95% test accuracy on the breast cancer dataset.")

## Next
Back to the path: https://ihor.sh/ai/ · next: chapter 4, backprop.